# `fathan/ijelid-ft-indojave-indobertweet` fine-tuning, then CatBoost on embeddings + keyword features

In [ ]:
DATA_DIR = "../data"

In [ ]:
import pandas as pd
import numpy as np
import re

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

import tensorflow as tf
from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())

True


In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train_cleaned_v3_keywords.csv')
train.head()

,text,label,pendidikan rendah,kelompok etnis,masyarakat lintas etnis,kaum difabel,bugis,generasi emas,low educated,menengah bawah,...,tata kata,ryan_fighter7,label_pol,label_sos,label_eko,label_aman,label_ide,label_sda,label_demo,label_geo
0,Kunjungan Prabowo ini untuk meresmikan dan men...,Sumber Daya Alam,0,0,0,0,0,0,0,0,...,0,0,Other,Other,Other,Other,Other,Sumber Daya Alam,Other,Other
1,Anies dapat tepuk tangan meriah saat jadi Rekt...,Politik,0,0,0,0,0,0,0,0,...,0,0,Politik,Other,Other,Other,Other,Other,Other,Other
2,emng bener sih pendukung 01 ada yang goblok be...,Demografi,1,0,0,0,0,0,0,0,...,0,0,Other,Other,Other,Other,Other,Other,Demografi,Other
3,Sewaktu anies bersikap kritis ke kinerja pak p...,Politik,0,0,0,0,0,0,0,0,...,0,0,Politik,Other,Other,Other,Other,Other,Other,Other
4,Anies Baswedan Harap aparatur sipil negara ter...,Politik,0,0,0,0,0,0,0,0,...,0,0,Politik,Other,Other,Other,Other,Other,Other,Other


In [ ]:
# Create a label mapping
label_mapping = {
    'Politik': 0,
    'Sosial Budaya': 1,
    'Ekonomi': 2,
    'Pertahanan dan Keamanan': 3,
    'Ideologi': 4,
    'Sumber Daya Alam': 5,
    'Demografi': 6,
    'Geografi': 7
}

inv_label_mapping = {v: k for k, v in label_mapping.items()}

# Apply the label mapping
train['label'] = train['label'].map(label_mapping)

In [ ]:
kw_df = pd.read_excel(f"{DATA_DIR}/keywords_prob.xlsx")
kw_df.head()

,keyword,Demografi,Ekonomi,Geografi,Ideologi,Pertahanan dan Keamanan,Sosial Budaya,Sumber Daya Alam,Politik
0,pendidikan rendah,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,kelompok etnis,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,masyarakat lintas etnis,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,kaum difabel,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,bugis,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Split the DataFrame into training and validation sets
train_df, val_df = train_test_split(train[['text','label'] + kw_df["keyword"].to_list()], test_size=0.2, stratify=train['label'], random_state=42)
train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

# Convert DataFrames to Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Combine into a DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})

In [ ]:
import torch
import gc


# Mengosongkan cache CUDA
torch.cuda.empty_cache()

# Mengumpulkan garbage
gc.collect()

# Memeriksa penggunaan memori CUDA
print(f"Memory Reserved: {torch.cuda.memory_reserved()} bytes")
print(f"Memory Allocated: {torch.cuda.memory_allocated()} bytes")
print(f"Max Memory Reserved: {torch.cuda.max_memory_reserved()} bytes")

# Restart kernel/environment jika diperlukan

Memory Reserved: 1799356416 bytes
Memory Allocated: 1348039168 bytes
Max Memory Reserved: 9281994752 bytes


In [ ]:
import numpy as np
from collections import Counter

# Assuming 'dataset' is your dataset and 'train' split contains the labels
labels = dataset['train']['label']
class_counts = Counter(labels)
total_samples = len(labels)

# Calculate weights for each class
class_weights = {cls: total_samples/count for cls, count in class_counts.items()}

# Normalize weights to sum to 1
sum_weights = sum(class_weights.values())
class_weights = {cls: weight/sum_weights for cls, weight in class_weights.items()}

# Convert weights to a list
class_weight_list = [class_weights[i] for i in range(len(class_weights))]

class_weight_list

[0.0036326804073170452,
 0.026910514069993372,
 0.03877154633781036,
 0.03971260328775721,
 0.04049899147167319,
 0.0802038850713528,
 0.1859271881199542,
 0.5843425912341418]

In [ ]:
# Define the model name
model_name = "fathan/ijelid-ft-indojave-indobertweet"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the tokenize function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, ignore_mismatched_sizes=True)

# Define a data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/3335 [00:00<?, ? examples/s]

Map:   0%|          | 0/834 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at fathan/ijelid-ft-indojave-indobertweet and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at fathan/ijelid-ft-indojave-indobertweet and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([7, 768]) in the checkpoint and torch.Siz

In [ ]:
import torch
from transformers import Trainer, TrainingArguments
from sklearn.metrics import balanced_accuracy_score

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Convert class weights to a tensor
        class_weights_tensor = torch.tensor(class_weight_list).to(model.parameters().__next__().device)

        # Compute weighted cross-entropy loss
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Define training arguments with evaluation and saving steps
training_args = TrainingArguments(
    output_dir=f"./{model_name}_results",
    evaluation_strategy="steps",
    save_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)


# Use the custom trainer for training
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
)

# Train the model
trainer.train()

# Make predictions on the validation set
predictions = trainer.predict(tokenized_datasets['validation'])

# Extract the predicted labels and true labels
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(labels, preds)
print("Balanced Accuracy:", balanced_acc)

# Clear CUDA cache if necessary
torch.cuda.empty_cache()

/opt/conda/lib/python3.10/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

  ········································


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss
100,1.868500,1.744033
200,1.509100,1.385976
300,1.070400,1.309093
400,1.047600,1.283374
500,0.790700,1.272002


/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/opt/conda/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead u

Balanced Accuracy: 0.5312511764086054


In [ ]:
from tqdm import tqdm

model = AutoModelForSequenceClassification.from_pretrained('fathan/ijelid-ft-indojave-indobertweet_results/checkpoint-500', num_labels=8)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Function to get embeddings using the fine-tuned model
def get_embeddings(texts, tokenizer, model, max_length=512):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Use the embeddings from the last hidden state (average pooling)
    embeddings = outputs.hidden_states[-1].mean(dim=1).cpu().numpy()
    return embeddings

# Assuming your train dataset is in a pandas DataFrame
train_texts = train_df['text'].tolist()

# Get embeddings for the train dataset
train_embeddings = []
for text in tqdm(train_texts, desc="Generating train embeddings"):
    embedding = get_embeddings([text], tokenizer, model)
    train_embeddings.append(embedding)

train_embeddings = np.vstack(train_embeddings)

# Convert embeddings to DataFrame and add as new features to the original dataset
train_embedding_df = pd.DataFrame(train_embeddings)
train_dataset_with_embeddings = pd.concat([train_df, train_embedding_df], axis=1)

# val
val_texts = val_df['text'].tolist()
val_embeddings = []
for text in tqdm(val_texts, desc="Generating val embeddings"):
    embedding = get_embeddings([text], tokenizer, model)
    val_embeddings.append(embedding)

val_embeddings = np.vstack(val_embeddings)
val_embedding_df = pd.DataFrame(val_embeddings)
val_dataset_with_embeddings = pd.concat([val_df, val_embedding_df], axis=1)

# Print the shape to verify
print(train_dataset_with_embeddings.shape)
print(val_dataset_with_embeddings.shape)

(3335, 770)
(834, 770)


In [ ]:
class_weights_dict = dict()
n, num_class = train_dataset_with_embeddings.shape[0], len(train_dataset_with_embeddings['label'].unique())
for label in train_dataset_with_embeddings['label'].unique():
    n_l = train_dataset_with_embeddings[train_dataset_with_embeddings['label'] == label].shape[0]
    if n_l == 0:
        continue
    class_weights_dict[label] = n / (num_class * n_l)

class_weights_dict

{0: 0.1851132326820604,
 1: 1.371299342105263,
 2: 1.9757109004739337,
 3: 2.0236650485436893,
 5: 4.0870098039215685,
 6: 9.474431818181818,
 7: 29.776785714285715,
 4: 2.063737623762376}

In [ ]:
train_dataset_with_embeddings = pd.concat([train_dataset_with_embeddings, train_df[kw_df["keyword"].to_list()]], axis=1)
val_dataset_with_embeddings = pd.concat([val_dataset_with_embeddings, val_df[kw_df["keyword"].to_list()]], axis=1)
train_dataset_with_embeddings.head()

,text,label,0,1,2,3,4,5,6,7,...,nyerocos,manyargubi,politik merdeka,futrijasun,vidaanasution,imadarch,chairudinn6548,sering ngibul,tata kata,ryan_fighter7
0,Joget joget menyenangkan tapi pemuda lebih mem...,0,0.197711,-0.856437,0.142392,-0.471621,-0.634158,-0.374566,-0.004935,-0.579650,...,0,0,0,0,0,0,0,0,0,0
1,Ganjar Mahfud membangun sinergi antarnegara AS...,0,0.421002,-0.852400,0.931399,-0.515352,0.206030,-0.529462,0.177435,-0.392608,...,0,0,0,0,0,0,0,0,0,0
2,Yuk pemilih Ganjar monggo di sah kan kosa kata...,1,0.252622,-0.454806,0.414912,-0.397153,-0.872411,0.079025,1.140562,0.616918,...,0,0,0,0,0,0,0,0,0,0
3,Kontribusi Pasangan Capres Cawapres Ganjar Pra...,2,0.416624,-0.415443,0.253617,0.261858,-0.406729,-0.319850,1.014517,0.354613,...,0,0,0,0,0,0,0,0,0,0
4,Suasana optimis melingkupi Jawa Tengah saat ma...,0,0.353465,-0.431591,-0.217194,-0.558609,-0.024744,-0.424450,0.305432,-0.478439,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
train_pool = Pool(
    # data=train_dataset_with_embeddings.drop("label", axis=1).to_numpy(),
    data=train_dataset_with_embeddings.drop(['label','text'], axis=1).to_numpy(),
    label=train_dataset_with_embeddings['label'].to_numpy(),
    feature_names=train_dataset_with_embeddings.drop(['label','text'], axis=1).columns.to_list()
)

val_pool = Pool(
    # data=val_df_split.drop("label", axis=1).to_numpy(),
    data=val_dataset_with_embeddings.drop(['label','text'], axis=1).to_numpy(),
    label=val_dataset_with_embeddings['label'].to_numpy(),
    feature_names=val_dataset_with_embeddings.drop(['label','text'], axis=1).columns.to_list()
)

In [ ]:
cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    task_type="GPU",
    devices="0",
    verbose=100,
    random_state=42,
    class_weights=class_weights_dict,
    iterations = 1000,
    learning_rate = 0.01
)

cb_model.fit(train_pool, eval_set=val_pool, plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 0.6805561	test: 0.4356872	best: 0.4356872 (0)	total: 93.3ms	remaining: 1m 33s
100:	learn: 0.8632483	test: 0.4861978	best: 0.5330437 (12)	total: 4.16s	remaining: 37s
200:	learn: 0.8810357	test: 0.4759825	best: 0.5330437 (12)	total: 8.35s	remaining: 33.2s
300:	learn: 0.8924617	test: 0.4727383	best: 0.5330437 (12)	total: 12.4s	remaining: 28.8s
400:	learn: 0.9032187	test: 0.4809720	best: 0.5330437 (12)	total: 16s	remaining: 23.9s
500:	learn: 0.9087198	test: 0.4820355	best: 0.5330437 (12)	total: 19.3s	remaining: 19.2s
600:	learn: 0.9125259	test: 0.4773290	best: 0.5330437 (12)	total: 22.4s	remaining: 14.9s
700:	learn: 0.9155653	test: 0.4788731	best: 0.5330437 (12)	total: 25.4s	remaining: 10.8s
800:	learn: 0.9216899	test: 0.4811506	best: 0.5330437 (12)	total: 28.4s	remaining: 7.06s
900:	learn: 0.9252140	test: 0.4823269	best: 0.5330437 (12)	total: 31.4s	remaining: 3.46s
999:	learn: 0.9259046	test: 0.4824401	best: 0.5330437 (12)	total: 34.4s	remaining: 0us
bestTest = 0.53304374
bestIt

In [ ]:
# Only BERT embeddings
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_dataset_with_embeddings['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.5390012505241271


In [ ]:
# Only BERT embeddings
predictions = cb_model.predict(val_pool)

# Calculate the balanced accuracy
balanced_acc = balanced_accuracy_score(val_dataset_with_embeddings['label'], predictions)
print("Balanced Accuracy:", balanced_acc)

Balanced Accuracy: 0.5390012505241271


In [ ]:
!zip ijelid_ft_indojave_indobertweet_results.zip fathan/ijelid-ft-indojave-indobertweet_results/checkpoint-500 -r

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  adding: kaggle/working/fathan/ijelid-ft-indojave-indobertweet_results/checkpoint-500/ (stored 0%)
  adding: kaggle/working/fathan/ijelid-ft-indojave-indobertweet_results/checkpoint-500/config.json (deflated 56%)
  adding: kaggle/working/fathan/ijelid-ft-indojave-indobertweet_results/checkpoint-500/scheduler.pt (deflated 55%)
  adding: kaggle/working/fathan/ijelid-ft-indojave-indobertweet_results/checkpoint-500/model.safetensors (deflated 7%)
  adding: kaggle/working/fathan/ijelid-ft-indojave-indobertweet_results/checkpoint-500/training_args.bin (deflated 51%)
  adding: kaggle/working/fathan/ijelid-ft-indojave-indobertweet_results/checkpoint-500/rng_state.pth (deflated 25%)
  adding: kaggle/working/fathan/ijelid-ft-indojave-indobertweet_results/checkpoint-500/trainer_state.json (deflated 79%)
  adding: kaggle/working/fathan/ijelid-ft-indojave-indobertweet_results/checkpoint-500/optimizer.pt (deflated 21%)


In [ ]:
from IPython.display import FileLink, display

# Save the model
display(FileLink(f'ijelid_ft_indojave_indobertweet_results.zip'))

/kaggle/working/ijelid_ft_indojave_indobertweet_results.zip